# 17.4 零點偏移有什麼作用？


四位元有16種編碼，若原數值範圍是-1到2，為何一定要平均分給負側和正側？可以把16個整數碼0至15對應到這段不對稱範圍，另外記住「哪個整數碼代表浮點零」。這個整數碼叫zero-point（零點）。前置是[刻度與反量化](https://birdhackor.github.io/tiny-perceptron-vlm/17.2.html)：scale決定相鄰格差多少，zero-point決定浮點零落在哪一格。

取 `[-1,0,1,2]`，範圍寬3，用0至15之間十五個間隔，scale是 `3/15=0.2`。-1應落在碼0，浮點零比-1多五格，所以zero-point=5；1落在10，2落在15。保存 `[0,5,10,15]`後，還原先減5再乘0.2，恰好得到原值。注意整數碼0對應浮點-1，而不是浮點零；「零點」名字指浮點0對應的碼5。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

x = torch.tensor([-1.0, 0.0, 1.0, 2.0])
low = min(x.min().item(), 0.0)
high = max(x.max().item(), 0.0)
scale = (high - low) / 15
zero = round(-low / scale)
q = (x / scale + zero).round().clamp(0, 15).to(torch.uint8)
restored = (q.float() - zero) * scale
print("scale/zero", scale, zero)
print("整數碼", q.tolist())
print("還原", restored.round(decimals=4).tolist())

`min`與`max`讓範圍包含浮點零，即使資料全部在2至3，也不能只依那個窄區間任意平移：否則浮點零可能沒有合法整數碼。`zero`用最近整數取得；`q`先縮放、加零點再取整與限制範圍。uint8表示不含負數的八位元整數容器，此時只使用0至15的16種碼，尚未把兩個四位元碼裝進一byte。這裡是表示規則，不是完整檔案打包。

輸出scale約0.2、zero為5，整數碼 `[0,5,10,15]`，還原 `[-1,0,1,2]`。此例剛好落在刻度上，所以沒有捨入誤差；一般數字仍會有。帶刻度與零點的方式常叫affine quantization（仿射量化），相對於把零固定在整數零、正負對稱分格的方式，多了一個可保存的偏移。

能準確還原浮點零很實用，因為補齊或空白特徵常使用零；但這不表示所有非零值都精確，也不表示所有量化實作採unsigned碼。本章後面的int4打包使用-8至7的signed數字，再映到儲存碼，必須知道各層在使用哪種表示，不能看到「4-bit」就直接混用。

練習把x改成 `[-2,0,1]`。先預測scale仍0.2，zero移到10，整數碼為 `[0,10,15]`，再執行核對。範圍寬度不變，刻度不變，但零的位置不同；這正是scale與zero-point分工的差別。
